# Bank Marketing – Data Analysis (Phases 1–3)

## Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

In [ ]:
# Use the local file when running inside the repo, otherwise read from GitHub (Colab)
LOCAL_PATH = Path('../dataset/bank/bank-full.csv')
DATASET_URL = 'https://raw.githubusercontent.com/MinhTuan2405/bankMarAnalysis/main/dataset/bank/bank-full.csv'
DATA_SOURCE = LOCAL_PATH if LOCAL_PATH.exists() else DATASET_URL
DATASET_SEPARATOR = ';'

TARGET = 'y'
RANDOM_STATE = 42
TEST_SIZE = 0.30

In [ ]:
def pct(x, d=2):
    return f'{x * 100:.{d}f}%'

# Phase 1. Data Collection and Understanding
## Step 1. Load the data

In [ ]:
df_raw = pd.read_csv(DATA_SOURCE, sep=DATASET_SEPARATOR)
print(f'Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}')
df_raw.dtypes.to_frame('dtype').T

In [ ]:
df_raw.head(10)

In [ ]:
FEATURES = [c for c in df_raw.columns if c != TARGET]
print('Target  :', TARGET, '->', df_raw[TARGET].unique().tolist())
print(f'Inputs ({len(FEATURES)}):', FEATURES)

## Step 2. Explore the data structure

In [ ]:
# (group, meaning, when the value is known)
VAR_INFO = {
    'age':       ('Client', 'Age', 'Before the call'),
    'job':       ('Client', 'Type of job', 'Before the call'),
    'marital':   ('Client', 'Marital status', 'Before the call'),
    'education': ('Client', 'Education level', 'Before the call'),
    'default':   ('Client', 'Has credit in default', 'Before the call'),
    'balance':   ('Client', 'Average yearly balance (EUR)', 'Before the call'),
    'housing':   ('Client', 'Has a housing loan', 'Before the call'),
    'loan':      ('Client', 'Has a personal loan', 'Before the call'),
    'contact':   ('Last contact', 'Contact communication type', 'Before the call'),
    'day':       ('Last contact', 'Day of month of the call', 'Before the call (schedule)'),
    'month':     ('Last contact', 'Month of the call', 'Before the call (schedule)'),
    'duration':  ('Last contact', 'Call duration (seconds)', 'After the call -> leakage'),
    'campaign':  ('Campaign', 'Contacts in this campaign (incl. last one)', 'Before the call'),
    'pdays':     ('Campaign', 'Days since previous campaign contact (-1 = never)', 'Before the call'),
    'previous':  ('Campaign', 'Contacts before this campaign', 'Before the call'),
    'poutcome':  ('Campaign', 'Outcome of the previous campaign', 'Before the call'),
    'y':         ('Target', 'Subscribed to a term deposit', 'Outcome'),
}

NUM_COLS = df_raw.select_dtypes('number').columns.tolist()
CAT_COLS = [c for c in df_raw.select_dtypes(exclude='number').columns if c != TARGET]

def var_kind(c):
    if c in NUM_COLS:
        return 'Numeric'
    return 'Binary' if df_raw[c].nunique() == 2 else 'Categorical'

var_table = pd.DataFrame({
    'group': [VAR_INFO[c][0] for c in df_raw.columns],
    'meaning': [VAR_INFO[c][1] for c in df_raw.columns],
    'kind': [var_kind(c) for c in df_raw.columns],
    'dtype': df_raw.dtypes.astype(str).values,
    'n_unique': df_raw.nunique().values,
    'examples': [', '.join(map(str, df_raw[c].unique()[:4])) for c in df_raw.columns],
    'known_when': [VAR_INFO[c][2] for c in df_raw.columns],
}, index=df_raw.columns)
var_table

In [ ]:
print('Numeric    :', NUM_COLS)
print('Categorical:', CAT_COLS)

In [ ]:
num_desc = df_raw[NUM_COLS].describe().T
num_desc['skew'] = df_raw[NUM_COLS].skew()
num_desc['%zero'] = (df_raw[NUM_COLS] == 0).mean() * 100
num_desc['%negative'] = (df_raw[NUM_COLS] < 0).mean() * 100
num_desc

In [ ]:
df_raw[CAT_COLS + [TARGET]].describe().T

In [ ]:
# Value distribution of categorical variables
cat_dist = pd.concat(
    {c: pd.DataFrame({'count': df_raw[c].value_counts(),
                      '%': df_raw[c].value_counts(normalize=True) * 100})
     for c in CAT_COLS + [TARGET]}
)
cat_dist

In [ ]:
# Dataset overview
target_rate = (df_raw[TARGET] == 'yes').mean()
overview = pd.Series({
    'Records': f'{len(df_raw):,}',
    'Input variables': len(FEATURES),
    'Numeric': len(NUM_COLS),
    'Categorical / binary': len(CAT_COLS),
    'Target': TARGET,
    'y = yes rate': pct(target_rate),
    'Missing cells (NaN)': int(df_raw.isna().sum().sum()),
    'Duplicate rows': int(df_raw.duplicated().sum()),
    'Memory (MB)': round(df_raw.memory_usage(deep=True).sum() / 1024**2, 2),
}, name='value').to_frame()
overview

In [ ]:
# Issues to handle (detected automatically)
unknown_cols = [c for c in CAT_COLS if (df_raw[c] == 'unknown').any()]
skewed_cols = num_desc.index[num_desc['skew'].abs() > 2].tolist()
pdays_neg = (df_raw['pdays'] == -1).mean()

issues = pd.DataFrame([
    ('Class imbalance', f'y = yes is only {pct(target_rate)}'),
    ("'unknown' values", ', '.join(unknown_cols)),
    ('Special value', f'pdays = -1 in {pct(pdays_neg)} of rows'),
    ('Heavy skew (|skew| > 2)', ', '.join(skewed_cols)),
    ('Negative values', f"balance < 0 in {pct((df_raw['balance'] < 0).mean())} of rows"),
    ('Data leakage', 'duration is only known after the call'),
], columns=['issue', 'detail'])
issues

# Phase 2. Data Preprocessing
## Step 3. Check and clean the data

In [ ]:
missing = pd.DataFrame({'missing': df_raw.isna().sum(), '%': df_raw.isna().mean() * 100})
missing.T

In [ ]:
unknown = pd.DataFrame({'unknown': (df_raw[CAT_COLS] == 'unknown').sum(),
                        '%': (df_raw[CAT_COLS] == 'unknown').mean() * 100})
unknown = unknown[unknown['unknown'] > 0].sort_values('%', ascending=False)
unknown

In [ ]:
# Subscription rate of 'unknown' vs known values -> does 'unknown' carry information?
unknown_rate = pd.DataFrame({
    'rate_unknown': [(df_raw.loc[df_raw[c] == 'unknown', TARGET] == 'yes').mean() for c in unknown.index],
    'rate_known': [(df_raw.loc[df_raw[c] != 'unknown', TARGET] == 'yes').mean() for c in unknown.index],
}, index=unknown.index)
unknown_rate

In [ ]:
n_dup = df_raw.duplicated().sum()
n_dup_x = df_raw.drop(columns=TARGET).duplicated().sum()
print(f'Fully duplicated rows: {n_dup} | Duplicated inputs (ignoring y): {n_dup_x}')

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < lo) | (s > hi)
    return pd.Series({'Q1': q1, 'Q3': q3, 'IQR': iqr, 'lower': lo, 'upper': hi,
                      'n_outliers': int(mask.sum()), '%outliers': mask.mean() * 100,
                      'min': s.min(), 'max': s.max()})

outlier_table = df_raw[NUM_COLS].apply(iqr_outliers).T
outlier_table

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, c in zip(axes.ravel(), NUM_COLS):
    sns.boxplot(x=df_raw[c], ax=ax, color='#4C72B0', flierprops={'markersize': 2})
    ax.set_title(c)
for ax in axes.ravel()[len(NUM_COLS):]:
    ax.remove()
plt.suptitle('Boxplots of numeric variables (raw data)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Extreme values worth a closer look
df_raw.loc[(df_raw['previous'] > 50) | (df_raw['campaign'] > 50)]

In [ ]:
# Is pdays = -1 consistent with previous = 0 and poutcome = 'unknown'?
no_prev = df_raw['pdays'] == -1
print(f'pdays = -1: {no_prev.sum():,} rows ({pct(no_prev.mean())})')
print(f"Agreement (pdays = -1) <=> (previous = 0): {pct((no_prev == (df_raw['previous'] == 0)).mean())}")
pd.crosstab(no_prev.rename('pdays == -1'), df_raw['poutcome'], margins=True)

In [ ]:
df_raw.loc[~no_prev, 'pdays'].describe().to_frame('pdays (pdays != -1)').T

In [ ]:
# Decisions on abnormal data
decisions = pd.DataFrame([
    ('NaN', 'None found', 'Nothing to do; a SimpleImputer is still kept in the Pipeline for safety'),
    ("'unknown' (job, education, contact, poutcome)", 'Keep as its own category',
     "Subscription rate of 'unknown' differs from the rest → informative; for poutcome/contact it mostly means not applicable"),
    ('Duplicate rows', 'Drop if any', 'Avoid double weighting and train/test leakage'),
    ('Outliers (balance, campaign, previous, ...)', 'Keep',
     'Valid real-world values; skewness handled by a log transform in the Pipeline (phase 5)'),
    ('pdays = -1', 'Keep for EDA; NaN + indicator for modelling', '-1 is a "never contacted" code, not a number of days'),
    ('duration', 'Exclude from the prediction model', 'Only known after the call → data leakage'),
], columns=['issue', 'decision', 'reason'])
decisions

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f'After cleaning: {df.shape[0]:,} rows x {df.shape[1]} columns')

## Step 4. Define the data used for prediction

In [ ]:
df[TARGET] = df[TARGET].map({'yes': 1, 'no': 0}).astype(int)
df[TARGET].value_counts().to_frame('count').T

In [ ]:
LEAKAGE_COLS = ['duration']
VALID_FEATURES = [c for c in FEATURES if c not in LEAKAGE_COLS]

feature_status = var_table.loc[FEATURES, ['group', 'kind', 'known_when']].copy()
feature_status['used_in_model'] = np.where(feature_status.index.isin(LEAKAGE_COLS), 'No (leakage)', 'Yes')
feature_status

In [ ]:
print(f'Valid features ({len(VALID_FEATURES)}):', VALID_FEATURES)

In [ ]:
# 70/30 stratified split (the predict notebook reproduces the same split)
X = df[VALID_FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

split_info = pd.DataFrame({
    'rows': [len(X_train), len(X_test)],
    'share': [len(X_train) / len(X), len(X_test) / len(X)],
    'y = 1 rate': [y_train.mean(), y_test.mean()],
}, index=['train', 'test'])
split_info

In [ ]:
# EDA below uses the training set only
train_df = X_train.join(y_train)

In [ ]:
# duration leakage illustration (train only)
dur = df.loc[X_train.index, 'duration']
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(x=y_train, y=dur, ax=ax, showfliers=False)
ax.set(xlabel='y', ylabel='duration (seconds)', title='duration by y (train)')
plt.show()
dur_corr = np.corrcoef(dur, y_train)[0, 1]
dur.groupby(y_train).median().to_frame('median duration')

# Phase 3. Exploratory Data Analysis (EDA) – training set
## Step 5. Distribution analysis

In [ ]:
target_counts = train_df[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['no (0)', 'yes (1)'], target_counts.values, color=['#8C8C8C', '#DD8452'])
for b, v in zip(bars, target_counts.values):
    ax.annotate(f'{v:,}\n({v / target_counts.sum():.1%})', (b.get_x() + b.get_width() / 2, v),
                ha='center', va='bottom')
ax.set(title='Target distribution (train)', ylabel='Clients')
ax.set_ylim(0, target_counts.max() * 1.15)
plt.show()

In [ ]:
EDA_NUM = ['age', 'balance', 'campaign', 'previous']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, c in enumerate(EDA_NUM):
    lo, hi = train_df[c].quantile([0.005, 0.995])
    view = train_df[train_df[c].between(lo, hi)]
    sns.histplot(data=view, x=c, hue=TARGET, stat='density', common_norm=False,
                 bins=40, element='step', ax=axes[0, i])
    axes[0, i].set_title(f'Histogram of {c} (0.5%–99.5%)')
    sns.boxplot(data=train_df, x=TARGET, y=c, ax=axes[1, i], showfliers=False)
    axes[1, i].set_title(f'{c} by y (outliers hidden)')
plt.tight_layout()
plt.show()

In [ ]:
num_by_y = train_df.groupby(TARGET)[EDA_NUM].agg(['mean', 'median']).T.rename(columns={0: 'no', 1: 'yes'})
num_by_y

In [ ]:
EDA_CAT = ['job', 'education', 'marital', 'housing', 'loan']

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, c in zip(axes.ravel(), EDA_CAT):
    order = train_df[c].value_counts().index
    sns.countplot(data=train_df, y=c, order=order, ax=ax, color='#4C72B0')
    ax.set_title(f'Clients by {c}')
axes.ravel()[-1].remove()
plt.tight_layout()
plt.show()

## Step 6. Relationship with the target

In [ ]:
OVERALL_RATE = train_df[TARGET].mean()

def rate_table(data, col):
    t = data.groupby(col, observed=True)[TARGET].agg(n='size', n_yes='sum', rate='mean')
    t['share'] = t['n'] / t['n'].sum()
    t['lift'] = t['rate'] / OVERALL_RATE
    return t.sort_values('rate', ascending=False)

def plot_rate(data, col, ax=None, sort=True):
    t = rate_table(data, col)                      # always returned sorted by rate
    p = t if sort else t.sort_index()              # display order on the chart
    ax = ax or plt.subplots(figsize=(7, 0.4 * len(p) + 1.5))[1]
    sns.barplot(x=(p['rate'] * 100).values, y=p.index.astype(str).values, ax=ax, color='#DD8452', orient='h')
    ax.axvline(OVERALL_RATE * 100, ls='--', c='k', lw=1, label=f'Overall {OVERALL_RATE:.1%}')
    for i, (r, n) in enumerate(zip(p['rate'], p['n'])):
        ax.text(r * 100, i, f' {r:.1%} (n={n:,})', va='center', fontsize=9)
    ax.set(xlabel='Subscription rate (%)', ylabel=col, title=f'Subscription rate by {col}')
    ax.legend(loc='lower right')
    return t

In [ ]:
# Job, education, marital status
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
rates = {c: plot_rate(train_df, c, ax) for c, ax in zip(['job', 'education', 'marital'], axes)}
plt.tight_layout()
plt.show()
pd.concat(rates)

In [ ]:
# Loan status
fig, axes = plt.subplots(1, 3, figsize=(20, 3.5))
for c, ax in zip(['housing', 'loan', 'default'], axes):
    rates[c] = plot_rate(train_df, c, ax)
plt.tight_layout()
plt.show()
pd.concat({c: rates[c] for c in ['housing', 'loan', 'default']})

In [ ]:
loan_combo = train_df.pivot_table(index='housing', columns='loan', values=TARGET, aggfunc='mean')
fig, ax = plt.subplots(figsize=(5, 3.5))
sns.heatmap(loan_combo * 100, annot=True, fmt='.1f', cmap='Oranges', ax=ax)
ax.set_title('Subscription rate (%) by housing x loan')
plt.show()

In [ ]:
# Campaign history: campaign, previous, pdays, poutcome
hist_df = train_df.assign(
    campaign_grp=pd.cut(train_df['campaign'], [0, 1, 2, 3, 4, 5, 10, np.inf],
                        labels=['1', '2', '3', '4', '5', '6-10', '>10']),
    previous_grp=pd.cut(train_df['previous'], [-1, 0, 1, 2, 3, 5, 10, np.inf],
                        labels=['0', '1', '2', '3', '4-5', '6-10', '>10']),
    pdays_grp=pd.cut(train_df['pdays'], [-2, -1, 90, 180, 365, np.inf],
                     labels=['never', '<=90', '91-180', '181-365', '>365']),
)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for c, ax in zip(['campaign_grp', 'previous_grp', 'pdays_grp', 'poutcome'], axes.ravel()):
    rates[c] = plot_rate(hist_df, c, ax, sort=(c == 'poutcome'))
plt.tight_layout()
plt.show()
pd.concat({c: rates[c] for c in ['campaign_grp', 'previous_grp', 'pdays_grp', 'poutcome']})

In [ ]:
# Extra: contact, timing, age and balance groups
extra_df = train_df.assign(
    age_grp=pd.cut(train_df['age'], [0, 25, 30, 40, 50, 60, np.inf],
                   labels=['<=25', '26-30', '31-40', '41-50', '51-60', '>60']),
    balance_grp=pd.qcut(train_df['balance'], 5, duplicates='drop'),
)
MONTH_ORDER = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
extra_df['month'] = pd.Categorical(extra_df['month'], MONTH_ORDER, ordered=True)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for c, ax in zip(['contact', 'month', 'age_grp', 'balance_grp'], axes.ravel()):
    rates[c] = plot_rate(extra_df, c, ax, sort=(c == 'contact'))
plt.tight_layout()
plt.show()

In [ ]:
# Highest / lowest subscription groups (groups with at least 1% of clients)
all_groups = pd.concat(
    [t.assign(variable=c, group=t.index.astype(str)) for c, t in rates.items()], ignore_index=True
)[['variable', 'group', 'n', 'share', 'rate', 'lift']]
all_groups = all_groups[all_groups['share'] >= 0.01]
top_groups = all_groups.nlargest(10, 'rate')
bottom_groups = all_groups.nsmallest(10, 'rate')
display(top_groups.style.format({'share': '{:.1%}', 'rate': '{:.1%}', 'lift': '{:.2f}'}).set_caption('Top 10 highest groups'))
display(bottom_groups.style.format({'share': '{:.1%}', 'rate': '{:.1%}', 'lift': '{:.2f}'}).set_caption('Top 10 lowest groups'))